In [1]:
import time
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# DATASET
(train_ds_raw, val_ds_raw), ds_info = tfds.load(
    "tf_flowers",
    split=["train[:80%]", "train[80%:]"],
    with_info=True,
    as_supervised=True,
    shuffle_files=True
)

num_classes = ds_info.features["label"].num_classes
class_names = ds_info.features["label"].names

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = preprocess_input(image)
    return image, label

train_ds = (
    train_ds_raw
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds_raw
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# AUGMENTATION
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# BEST MODEL: MobileNetV2 Transfer Learning
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

# FINE-TUNING
base_model.trainable = True

fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

training_time = time.time() - start_time

loss, accuracy = model.evaluate(val_ds)

print(f"Best model: MobileNetV2 Transfer Learning")
print(f"Validation accuracy: {accuracy:.4f}")
print(f"Training time: {training_time:.2f} sec")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/tf_flowers/incomplete.VG03EL_3.0.1/tf_flowers-train.tfrecord-[0-9][0-9][0-…

Dataset tf_flowers downloaded and prepared to /root/tensorflow_datasets/tf_flowers/3.0.1. Subsequent calls will reuse this data.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 21s 107ms/step - accuracy: 0.6625 - loss: 0.8860 - val_accuracy: 0.8515 - val_loss: 0.4463
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 9s 78ms/step - accuracy: 0.8338 - loss: 0.4785 - val_accuracy: 0.8951 - val_loss: 0.3478
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 8s 74ms/step - accuracy: 0.8583 - loss: 0.4010 - val_accuracy: 0.8910 - val_loss: 0.3121
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 85ms/step - accuracy: 0.8736 - loss: 0.3519 - val_accuracy: 0.9005 - val_loss: 0.2974
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 85ms/step - accuracy: 0.8883 - loss: 0.3236 - val_accuracy: 0.8951 - val_loss: 0.2962
Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 27s 134ms/step - accuracy: 0.7847 - loss: 0.5547 - val_accuracy: 0.8869 - val_loss: 0.3163
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 13s 122ms/step - accura